# Notebook 1 — News Scraper (Apify) v2

**What's new in v2:**
- Verified working sources only (CNN / WSJ / Forbes removed — confirmed stale in live tests)
- 48-hour date filter — drops any article older than 2 days automatically
- Similarity deduplication — removes near-duplicate stories using TF-IDF cosine similarity
- Section classification — assigns each article to: Tech, World, Politics, Business, Crime, Science, Lifestyle, Sport, Entertainment

**Output:** `smart_news_raw.csv`  
New columns vs v1: `section`, `age_hours`

In [ ]:
# %pip install apify-client pandas scikit-learn python-dateutil

In [ ]:
from apify_client import ApifyClient
import pandas as pd
from datetime import datetime, timezone
from dateutil import parser as dateparser
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('Libraries loaded.')

In [ ]:
# ═══════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════
APIFY_API_TOKEN        = 'YOUR_APIFY_API_TOKEN_HERE'
ACTOR_ID               = 'complex_intricate_networks/news-article-scraper-100-global-sources-api'
CATEGORY               = 'Top news'
MAX_ARTICLES_PER_SOURCE = 100
KEYWORD_FILTER         = ''
MAX_AGE_HOURS          = 48    # Drop articles older than this
SIMILARITY_THRESHOLD   = 0.70  # Cosine similarity cutoff for near-duplicate removal

# ── VERIFIED WORKING SOURCES (live-tested May 2026) ────────────────────
# REMOVED: CNN (returns Apr 2023 data), WSJ World News (Jan 2025), Forbes (Jan 2024)
SOURCES = [
    'BBC World News',
    'Al Jazeera',
    'CNBC',
    'The New York Times',
    'TechCrunch',
    'Wired',
    'Variety',
    'The Guardian World News',
    'Business Insider',
    'Mashable',
    'The Hollywood Reporter',
    'Rolling Stone',
    'Vox',
    'TIME',
    'The New Yorker',
    'Ars Technica',
    'CNET News',
    'Nature',
    'CBS News',
    'CoinDesk',
]

In [ ]:
# ═══════════════════════════════════════════════════════
# SECTION CLASSIFIER
# Keyword matching on title + description. No API needed.
# ═══════════════════════════════════════════════════════

SECTION_KEYWORDS = {
    'Tech': [
        'ai', 'artificial intelligence', 'machine learning', 'software', 'app',
        'startup', 'google', 'apple', 'microsoft', 'meta', 'openai', 'nvidia',
        'chip', 'semiconductor', 'robot', 'autonomous', 'cybersecurity', 'hack',
        'data breach', 'cloud', 'electric vehicle', 'ev', 'tesla', 'spacex',
        'smartphone', 'iphone', 'android', 'tech', 'technology', 'internet',
        'programming', 'developer', 'algorithm', 'chatgpt', 'llm',
        'drone', 'satellite', 'broadband', '5g', 'quantum', 'computer'
    ],
    'Politics': [
        'election', 'president', 'prime minister', 'parliament', 'congress',
        'senate', 'government', 'policy', 'democrat', 'republican', 'labour',
        'conservative', 'trump', 'biden', 'vote', 'ballot', 'referendum',
        'legislation', 'minister', 'cabinet', 'diplomat', 'sanctions', 'treaty',
        'nato', 'united nations', 'white house', 'political', 'politician',
        'campaign', 'opposition', 'albanese', 'dutton', 'liberal party'
    ],
    'Business': [
        'stock', 'market', 'shares', 'economy', 'inflation', 'interest rate',
        'reserve bank', 'federal reserve', 'gdp', 'recession', 'investment',
        'revenue', 'profit', 'earnings', 'merger', 'acquisition', 'ipo',
        'bankruptcy', 'layoff', 'jobs', 'unemployment', 'trade', 'tariff',
        'oil', 'finance', 'bank', 'crypto', 'bitcoin', 'ethereum',
        'venture capital', 'wall street', 'asx', 'nasdaq', 'real estate', 'property'
    ],
    'Crime': [
        'murder', 'killed', 'arrested', 'charged', 'sentenced', 'prison',
        'jail', 'court', 'trial', 'verdict', 'conviction', 'police',
        'detective', 'investigation', 'suspect', 'shooting', 'stabbing',
        'robbery', 'theft', 'fraud', 'scam', 'trafficking', 'drug bust',
        'cartel', 'gang', 'terrorist', 'bomb', 'hostage', 'kidnap',
        'homicide', 'assault', 'domestic violence', 'corruption', 'bribery'
    ],
    'Science': [
        'research', 'study', 'scientists', 'discovery', 'space', 'nasa',
        'climate change', 'environment', 'species', 'fossil', 'gene', 'dna',
        'vaccine', 'virus', 'bacteria', 'cancer', 'pandemic', 'outbreak',
        'disease', 'biology', 'physics', 'astronomy', 'planet', 'telescope',
        'laboratory', 'reef', 'earthquake', 'volcano', 'ocean', 'atmosphere'
    ],
    'Sport': [
        'match', 'tournament', 'championship', 'league', 'cup', 'final',
        'score', 'goal', 'player', 'coach', 'transfer', 'football', 'soccer',
        'rugby', 'cricket', 'tennis', 'golf', 'basketball', 'nba', 'nfl',
        'nrl', 'afl', 'f1', 'formula 1', 'olympics', 'world cup',
        'grand slam', 'wimbledon', 'athlete', 'stadium', 'penalty', 'motogp'
    ],
    'Entertainment': [
        'movie', 'film', 'box office', 'oscar', 'bafta', 'emmy', 'grammy',
        'celebrity', 'actor', 'actress', 'director', 'album', 'concert',
        'streaming', 'netflix', 'disney', 'hbo', 'tv show', 'series',
        'trailer', 'premiere', 'red carpet', 'award', 'singer', 'band',
        'music', 'pop', 'hip hop', 'influencer', 'viral', 'tiktok'
    ],
    'Lifestyle': [
        'food', 'recipe', 'restaurant', 'travel', 'holiday', 'vacation',
        'wellness', 'mental health', 'fitness', 'workout', 'diet', 'parenting',
        'relationship', 'dating', 'wedding', 'home', 'interior', 'garden',
        'fashion', 'beauty', 'skincare', 'shopping', 'review', 'how to',
        'guide', 'tips', 'advice', 'coffee', 'wine', 'coupon', 'promo'
    ],
    'World': [
        'war', 'conflict', 'military', 'troops', 'attack', 'strike',
        'missile', 'ceasefire', 'peace talks', 'humanitarian', 'refugee',
        'ukraine', 'russia', 'israel', 'gaza', 'iran', 'china', 'north korea',
        'middle east', 'africa', 'europe', 'asia', 'latin america',
        'flood', 'disaster', 'protest', 'revolution', 'coup', 'foreign'
    ],
}

# Priority order — first match wins when multiple sections apply.
# Crime and Tech are checked before the broader World bucket.
SECTION_PRIORITY = ['Crime', 'Tech', 'Politics', 'Business', 'Science',
                    'Sport', 'Entertainment', 'Lifestyle', 'World']

def classify_section(title, description):
    text = (title + ' ' + description).lower()
    for section in SECTION_PRIORITY:
        if any(kw in text for kw in SECTION_KEYWORDS[section]):
            return section
    return 'World'

print(f'Section classifier ready. Sections: {SECTION_PRIORITY}')

In [ ]:
# ═══════════════════════════════════════════════════════
# DATE UTILITIES
# ═══════════════════════════════════════════════════════

def parse_published_at(raw):
    if not raw or not isinstance(raw, str):
        return None
    try:
        dt = dateparser.parse(raw)
        if dt and dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt
    except Exception:
        return None

def age_hours(dt):
    if dt is None:
        return 9999
    return (datetime.now(timezone.utc) - dt).total_seconds() / 3600

print('Date utils ready.')

In [ ]:
# ═══════════════════════════════════════════════════════
# SIMILARITY DEDUPLICATOR (TF-IDF cosine)
# ═══════════════════════════════════════════════════════

def deduplicate_by_similarity(df, threshold=0.70):
    """
    Removes near-duplicate articles.
    Two articles are duplicates if their title+description
    cosine similarity >= threshold.
    Earlier article (lower index) is always kept.
    """
    if len(df) < 2:
        return df

    texts = (df['title'].fillna('') + ' ' + df['description'].fillna('')).tolist()
    vectorizer  = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2))
    tfidf       = vectorizer.fit_transform(texts)
    sim_matrix  = cosine_similarity(tfidf)

    to_drop = set()
    for i in range(len(df)):
        if i in to_drop:
            continue
        for j in range(i + 1, len(df)):
            if j not in to_drop and sim_matrix[i, j] >= threshold:
                to_drop.add(j)

    result = df.drop(index=list(to_drop)).reset_index(drop=True)
    print(f'  Similarity dedup ({threshold}): {len(df)} -> {len(result)} '
          f'({len(to_drop)} near-duplicates removed)')
    return result

print('Similarity deduplicator ready.')

In [ ]:
# ═══════════════════════════════════════════════════════
# CONTENT FILTERS (original notebook logic + promo filter)
# ═══════════════════════════════════════════════════════

BAD_URL_KEYWORDS = ['/video/', '/live-news/', '/category/', '/index',
                    '/watch', '/gallery/', '/podcast/', '/audio/', '/live/']
GENERIC_TITLES   = ['latest news', 'breaking news', 'weather video',
                    'page not found', 'just in']
PROMO_KEYWORDS   = ['promo code', 'coupon', '% off', 'discount',
                    'gift guide', "mother's day", "father's day"]

def is_valid(item):
    url   = (item.get('url') or '').lower()
    title = (item.get('title') or '').lower().strip()
    desc  = (item.get('description') or '')
    if any(kw in url for kw in BAD_URL_KEYWORDS):    return False, 'bad URL'
    if title in GENERIC_TITLES:                       return False, 'generic title'
    if len(desc) < 20:                                return False, 'no description'
    if any(kw in title for kw in PROMO_KEYWORDS):    return False, 'promo page'
    return True, 'ok'

print('Content filters ready.')

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 1 — RUN APIFY ACTOR
# ═══════════════════════════════════════════════════════

VALID_ACTOR_SOURCES = [
    'The Times of India','GeeksforGeeks','Fandom','Hindustan Times','Economic Times',
    'Moneycontrol','The Indian Express','Mint','The Hindu','Investopedia','India Today',
    'ESPN Cricinfo','NDTV','Healthline','Forbes','MakeUseOf','TechTarget','BBC World News',
    'Business Standard','The Guardian World News','News18','Gadgets 360','Business Today',
    'ABP News','GSMArena','The New York Times','HowToGeek','XDA Developers','TechRadar',
    'IGN Reviews','Nature','SamMobile','India.com','Lifewire','Screenrant','Bloomberg',
    'Gadgets Now','PCMag UK','Tech Advisor','CNET News','Business Insider','Game Rant',
    'Times Now','Techspot','India TV','Filmibeat','Al Jazeera','Collider','CNBC',
    'Techopedia','CBR',"Tom's Guide",'Windows Report','CNN','Amar Ujala','Prokerala',
    'Android Authority','Oneindia','Beebom','Deccan Herald','Anime News Network India',
    'WSJ World News','Zee Business','Variety','Goodreturns','CNBCTV18','Windows Central',
    'Very Well Mind','Harvard Business Review','The New Indian Express','Very Well Health',
    'Wired',"Tom's Hardware",'Washington Post World News','Analytics India Magazine',
    'Cosmopolitan','TheGamer','Myupchar','YourStory','Firstpost','Games Radar','Scroll.in',
    'The Windows Club','ThePrint','Financial Times','TIME','TechCrunch','Vogue India',
    'PC Gamer','DNA India','The Quint','Deccan Chronicle','Polygon','The Hollywood Reporter',
    "It's FOSS",'Pocket-lint','Pinkvilla','Mashable','CBS News','Bollywood Hungama','Digit',
    'Webdunia English','Dot Esports','Help Desk Geek','Eurogamer','The New Yorker','TecMint',
    'Rolling Stone','Ars Technica','CoinDesk','Search Engine Land','GQ India','Onmanorama',
    'Republic World','Fextralife','Free Press Journal','Daily Mail India','Vox',
    'Rock Paper Shotgun','ArchDaily','Deadline','Billboard','The Economist','Make Tech Easier',
    'The Week','The Atlantic','The News Minute','Welcome to Good Food',
    'Conde Nast Traveller India','Sportstar','Independent Asia','ANI','Brave Blog','ESPN India'
]

filtered_sources = [s for s in SOURCES if s in VALID_ACTOR_SOURCES]
skipped_sources  = [s for s in SOURCES if s not in VALID_ACTOR_SOURCES]
print(f'Sources ({len(filtered_sources)}): {filtered_sources}')
if skipped_sources:
    print(f'Skipped (not in Actor list): {skipped_sources}')

print('\nCalling Apify Actor...')
client = ApifyClient(APIFY_API_TOKEN)
run    = client.actor(ACTOR_ID).call(run_input={
    'category':           CATEGORY,
    'countries':          ['Australia'],
    'sources':            filtered_sources,
    'maxArticles':        MAX_ARTICLES_PER_SOURCE,
    'keywordFilter':      KEYWORD_FILTER,
    'proxyConfiguration': {'useApifyProxy': True}
})
items = list(client.dataset(run['defaultDatasetId']).iterate_items())
print(f'Apify returned {len(items)} raw articles.')

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 2 — CONTENT FILTER
# ═══════════════════════════════════════════════════════

scrape_time = datetime.now().strftime('%H:%M:%S')
raw_rows, filter_log = [], {}

for item in items:
    valid, reason = is_valid(item)
    filter_log[reason] = filter_log.get(reason, 0) + 1
    if not valid:
        continue
    pub_dt  = parse_published_at(item.get('published_at', ''))
    hrs_old = age_hours(pub_dt)
    raw_rows.append({
        'website_name': item.get('source', '').strip(),
        'title':        item.get('title', '').strip(),
        'description':  item.get('description', '').strip(),
        'url':          item.get('url', ''),
        'published_at': pub_dt.strftime('%Y-%m-%d %H:%M UTC') if pub_dt else '',
        'age_hours':    round(hrs_old, 1),
        'scrape_time':  scrape_time,
        'ai_summary':   '',
        'section':      '',
    })

print(f'After content filter : {len(raw_rows)} / {len(items)}')
print(f'Filter reasons       : {filter_log}')

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 3 — 48-HOUR DATE FILTER
# ═══════════════════════════════════════════════════════

df     = pd.DataFrame(raw_rows)
before = len(df)
df     = df[df['age_hours'] <= MAX_AGE_HOURS].reset_index(drop=True)
print(f'Date filter ({MAX_AGE_HOURS}h)  : {before} -> {len(df)} ({before - len(df)} stale articles dropped)')

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 4 — EXACT TITLE DEDUP
# ═══════════════════════════════════════════════════════

before = len(df)
df     = df.drop_duplicates(subset=['title']).reset_index(drop=True)
print(f'Exact dedup          : {before} -> {len(df)} ({before - len(df)} exact duplicates removed)')

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 5 — SIMILARITY DEDUP
# ═══════════════════════════════════════════════════════

df = deduplicate_by_similarity(df, threshold=SIMILARITY_THRESHOLD)

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 6 — SECTION CLASSIFICATION
# ═══════════════════════════════════════════════════════

df['section'] = df.apply(lambda r: classify_section(r['title'], r['description']), axis=1)

print('Section distribution:')
print(df['section'].value_counts().to_string())

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 7 — SAVE
# ═══════════════════════════════════════════════════════

col_order = ['website_name', 'section', 'title', 'description',
             'ai_summary', 'url', 'published_at', 'age_hours', 'scrape_time']
df = df[[c for c in col_order if c in df.columns]]
df.to_csv('smart_news_raw.csv', index=False, encoding='utf-8')

print(f'\nFinal: {len(df)} articles saved to smart_news_raw.csv')
print(f'Raw from Apify: {len(items)} | After all filters: {len(df)}')

In [ ]:
# ═══════════════════════════════════════════════════════
# OPTIONAL — PER-SECTION PREVIEW
# ═══════════════════════════════════════════════════════

for section in SECTION_PRIORITY:
    subset = df[df['section'] == section]
    if subset.empty:
        continue
    print(f'\n{"─"*60}')
    print(f'[{section.upper()}] — {len(subset)} articles')
    for _, row in subset.head(3).iterrows():
        print(f"  {row['title']}  [{row['website_name']}, {row['age_hours']}h ago]")